# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NAJAM2005/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

> **Data note.** This notebook uses the public 30,000-page starter CSV in the repo (32 clients, 54.2% of pages labelled declining). The capstone uses a different, gated lane cut (10,550 pages, 29 clients, 45.0% declining, separate impression windows), so numbers here are not directly comparable with the capstone paper. Both are observed, directional and for decision-support.

**What I see (n = 30,000 pages from the public starter slice; 32 clients; 54.2% labelled declining).** Every traffic field is heavy-tailed: impressions have a skew of 11.4 and the 99th percentile is about 100 times the median; search volume has a skew of 26.0, 8.2% missing and 40.2% zeros; CTR has a skew of 17.4 and 44.0% zeros. Days since last update looks bounded but piles up: 8,773 pages (29.2%) have exactly 104 days, which looks like a cap or default, not a real age. So I use buckets, ranks and logs, never raw averages.

In [1]:
import os, json, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
RAW = "https://raw.githubusercontent.com/NAJAM2005/flyrank-ml-internship/main/"
PATH = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(PATH if os.path.exists(PATH) else RAW + PATH)      # public starter slice (anonymised)
df["is_declining"] = (df["trend_direction"] == "down").astype(int)   # outcome for checks only, never a feature
print(df.shape, "| declining share:", round(df["is_declining"].mean(), 3), "| clients:", df["client_id"].nunique())

cols = ["days_since_last_update", "impressions_90d", "search_volume", "ctr", "avg_position", "word_count"]
rows = []
for c in cols:
    s = df[c].dropna()
    rows.append({"field": c, "n": len(s), "missing_%": round(100 * (1 - len(s) / len(df)), 1),
                 "median": round(s.median(), 2), "p90": round(s.quantile(.9), 2), "p99": round(s.quantile(.99), 2),
                 "max": round(s.max(), 2), "skew": round(s.skew(), 1), "zeros_%": round(100 * (s == 0).mean(), 1)})
print(pd.DataFrame(rows).to_string(index=False))
print("\np99 / median:", {c: round(df[c].quantile(.99) / max(df[c].median(), 1e-9), 1) for c in ["impressions_90d", "search_volume", "ctr"]})
n104 = int((df["days_since_last_update"] == 104).sum())
print("pages with days_since_last_update exactly 104:", n104, f"({n104 / len(df):.1%})")


(30000, 45) | declining share: 0.542 | clients: 32
                 field     n  missing_%  median      p90      p99      max  skew  zeros_%
days_since_last_update 30000        0.0   20.00   104.00   106.00    373.0   1.2      0.0
       impressions_90d 30000        0.0  731.00 12136.40 73505.83 517715.0  11.4      0.0
         search_volume 27532        8.2   10.00   110.00  2900.00  74000.0  26.0     40.2
                   ctr 30000        0.0    0.07     0.65     8.33    100.0  17.4     44.0
          avg_position 30000        0.0   10.80    36.80    69.90    245.0   2.0      4.0
            word_count 22301       25.7 2877.00  5327.00  7292.00   9546.0   0.9      0.0

p99 / median: {'impressions_90d': np.float64(100.6), 'search_volume': np.float64(290.0), 'ctr': np.float64(119.0)}
pages with days_since_last_update exactly 104: 8773 (29.2%)


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

**Three safe signals, each against the share of pages labelled declining (the label is used only to test, never as a feature).**

1. **Staleness (days since last update), n = 30,000: MIXED.** Decline rates by bucket are 53.9%, 39.3%, 59.9% and 54.7%, so there is no steady rise with age.
2. **Search volume, n = 27,532: CONFIRMED.** Decline rate falls steadily as volume rises: 59.0%, 52.5%, 49.8% (a 9.2-point spread). Caveat: 40% of pages have zero volume, which may mean unknown demand, so the lowest bucket mixes real low-demand pages with missing data.
3. **CTR versus peers at the same position, n = 25,932: MIXED.** Rates are 58.6%, 64.5%, 58.3%, 50.9%: lower CTR goes with more decline at the high end, but the lowest bucket breaks the pattern.

All of this is observed and directional.

In [2]:
def bucket_check(d, col, expect, q=5):
    x = d[[col, "is_declining"]].dropna().copy()
    x["bucket"] = pd.qcut(x[col], q, duplicates="drop")
    t = (x.groupby("bucket", observed=True)
           .agg(n=("is_declining", "size"), decline_rate=("is_declining", "mean")).round(3))
    print(f"{col}  (total n = {len(x)})")
    print(t.to_string())
    r, diffs = t["decline_rate"].values, np.diff(t["decline_rate"].values)
    if r.max() - r.min() < 0.03:
        v = "FALSE"
    elif (diffs >= 0).all() or (diffs <= 0).all():
        v = "CONFIRMED" if ((diffs >= 0).all() == (expect == "up")) else "OPPOSITE"
    else:
        v = "MIXED"
    print("verdict:", v, "| spread (max-min decline rate):", round(float(r.max() - r.min()), 3), "\n")
    return v

# Signal 1: staleness  (expect: more days since update -> more decline)
v1 = bucket_check(df, "days_since_last_update", expect="up")
# Signal 2: search volume (expect: higher volume -> less decline; demand cushions a page)
v2 = bucket_check(df, "search_volume", expect="down")
# Signal 3: CTR versus peers at the same position (expect: lower ratio -> more decline)
d = df[(df["avg_position"] > 0) & (df["impressions_90d"] > 0)].copy()
d["pos_bucket"] = pd.qcut(d["avg_position"], 10, duplicates="drop")
d["expected_ctr"] = d.groupby("pos_bucket", observed=True)["ctr"].transform("median")
d["ctr_ratio"] = d["ctr"] / d["expected_ctr"].replace(0, np.nan)
df["ctr_ratio"] = d["ctr_ratio"]
v3 = bucket_check(df, "ctr_ratio", expect="down")


days_since_last_update  (total n = 30000)
                    n  decline_rate
bucket                             
(0.999, 20.0]   15866         0.539
(20.0, 22.0]     3564         0.393
(22.0, 104.0]   10252         0.599
(104.0, 373.0]    318         0.547
verdict: MIXED | spread (max-min decline rate): 0.206 

search_volume  (total n = 27532)
                     n  decline_rate
bucket                              
(-0.001, 10.0]   18392         0.590
(10.0, 40.0]      4267         0.525
(40.0, 74000.0]   4873         0.498
verdict: CONFIRMED | spread (max-min decline rate): 0.092 

ctr_ratio  (total n = 25932)
                     n  decline_rate
bucket                              
(-0.001, 0.364]  10447         0.586
(0.364, 1.625]    5190         0.645
(1.625, 3.684]    5122         0.583
(3.684, 2500.0]   5173         0.509
verdict: MIXED | spread (max-min decline rate): 0.136 



## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

**Flag-linked test: staleness, the signal behind FlyRank's refresh flags.** The flags assume staler pages are the ones that fall. Pooled, the data seems to agree: pages not updated for more than 22 days decline 59.7% of the time (n = 10,570) against 51.2% for fresher pages (n = 19,430). But three things weaken it:

- Inside clients it mostly disappears: stale pages decline more in only **3 of 9** clients with enough pages (median gap -2.2 points).
- The 23-103 day group (49.7%) declines *less* than the freshest group; the whole pooled gap sits in the "exactly 104 days" group (61.6%).
- Those 104-day pages are not spread evenly: they sit in 14 clients, two clients hold 69% of them, and in 5 of 32 clients more than half of all pages are exactly 104 days.

**Verdict: MIXED.** The rule's assumption is not cleanly supported; the pooled effect largely reflects client composition and a 104-day value that may be a placeholder. This also tells me my Week-4 baseline (a stale-days flag) borrows strength from that group.

In [3]:
# Flag-linked test: FlyRank's refresh flags assume that staler pages are the ones that fall.
df["age_of_update_group"] = pd.cut(df["days_since_last_update"], [0, 22, 103, 104, 1000],
                                   labels=["1-22", "23-103", "exactly 104", "105+"])
t = df.groupby("age_of_update_group", observed=True)["is_declining"].agg(n="size", decline_rate="mean").round(3)
print(t.to_string())

stale = df["days_since_last_update"] > 22
print("\nstale (>22 days): n =", int(stale.sum()), "decline rate =", round(df.loc[stale, "is_declining"].mean(), 3))
print("fresh (<=22 days): n =", int((~stale).sum()), "decline rate =", round(df.loc[~stale, "is_declining"].mean(), 3))

# does the direction hold inside each client? (clients with at least 100 pages in both groups)
rows = []
for c, g in df.groupby("client_id"):
    a, b = g[g["days_since_last_update"] > 22], g[g["days_since_last_update"] <= 22]
    if len(a) >= 100 and len(b) >= 100:
        rows.append(a["is_declining"].mean() - b["is_declining"].mean())
rows = np.array(rows)
print(f"\nwithin clients: stale pages decline more in {int((rows > 0).sum())} of {len(rows)} clients (median gap {np.median(rows) * 100:+.1f} pts)")

at104 = df[df["days_since_last_update"] == 104]["client_id"].value_counts(normalize=True)
print(f"\n'exactly 104 days' pages sit in {at104.size} clients; the top 2 clients hold {at104.head(2).sum():.0%} of them, the top 1 holds {at104.iloc[0]:.0%}")
share_in_client = df.assign(is104=(df["days_since_last_update"] == 104)).groupby("client_id")["is104"].mean()
print("clients where more than half of pages are exactly 104 days:", int((share_in_client > 0.5).sum()), "of", len(share_in_client))


                         n  decline_rate
age_of_update_group                     
1-22                 19430         0.512
23-103                1479         0.497
exactly 104           8773         0.616
105+                   318         0.547

stale (>22 days): n = 10570 decline rate = 0.597
fresh (<=22 days): n = 19430 decline rate = 0.512



within clients: stale pages decline more in 3 of 9 clients (median gap -2.2 pts)

'exactly 104 days' pages sit in 14 clients; the top 2 clients hold 69% of them, the top 1 holds 35%
clients where more than half of pages are exactly 104 days: 5 of 32


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

**What a content team should take from this.** Use staleness as a prompt to look, not as proof that a page is falling: it is pooled evidence that weakens inside clients, and a large share of the "stale" pages carry a suspicious 104-day value. Search volume is the steadier cushion (low-volume pages slip more). Compare a page with other pages of the *same client*, and check what "104 days" really means before acting on it. Everything here is observed, directional and for decision-support.

In [4]:
summary = pd.DataFrame({
    "signal": ["days since last update", "search volume", "CTR vs peers at same position"],
    "verdict": [v1, v2, v3],
    "how to use it in triage": ["use a simple flag (>22 days), not a smooth score",
                                "use as a cushion: low-volume pages are the ones that slip",
                                "do not rank on it alone; the pattern is not monotone"]})
print(summary.to_string(index=False))


                       signal   verdict                                   how to use it in triage
       days since last update     MIXED          use a simple flag (>22 days), not a smooth score
                search volume CONFIRMED use as a cushion: low-volume pages are the ones that slip
CTR vs peers at same position     MIXED      do not rank on it alone; the pattern is not monotone


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.